# Operational Checks Workflow

This workbook illustrates operational validation checks that can be performed on valuation-date position data before it is used in risk calculations and reporting workflows. It runs SQL queries and Bloomberg-style market-data calls through helper functions, so the notebook shows the control flow without exposing all implementation details.

This is not a production workflow or a risk report. The purpose is to make the control logic visible: examples of checks that can be performed, where SQL queries are used, and where Bloomberg-style data is called in the background.

In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

from src.data.daily_workflow import (
    summarise_position_snapshot,
    validate_prices,
    build_cash_price_check,
    build_onboarding_review,
    build_market_context,
    build_exception_review,
    summarise_risk_ready_dataset,
    load_workflow_dates,
)

from src.data.enrichment import get_risk_ready_df

In [ ]:
# Set up the database, connect to database and market-data feeder
from src.setup_db import run as setup_db
setup_db()

import src.data.database as db
ENGINE = db.get_engine()

from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

**Dates used in the example**

`VALUATION_DATE` is imported from project configuration and represents the as-of date for positions, prices, NAV, and risk inputs. The notebook also derives `SIMULATED_RUN_DATE` as the next business day after `VALUATION_DATE`, only to show when this fixed simulation would be run by the AIFM team. In production, the workflow calendar would select the valuation date to process. In this notebook, the simulation is anchored on the project-wide `VALUATION_DATE` so the example remains reproducible.

In [ ]:
from src.config import VALUATION_DATE
workflow_dates = load_workflow_dates(VALUATION_DATE)

SIMULATED_RUN_DATE = workflow_dates["simulated_run_date"]
PRIOR_BUSINESS_DATE = workflow_dates["prior_business_date"]
LOOKBACK_START_DATE = workflow_dates["lookback_start_date"]

display(workflow_dates["dates_table"])

**Fund in this example**

In [ ]:
FUND_ID = "AIFM_HedgeFund"

## 1. Load valuation-date positions

Load positions for `VALUATION_DATE` and compare them with `PRIOR_BUSINESS_DATE`.

This comparison is used to identify new instruments, removed instruments, and portfolio-level changes before price validation and enrichment. `PRIOR_BUSINESS_DATE` is not a separate valuation date concept; it is only the prior business date used for comparison checks.

<small>

Background call:

```
SELECT *, position_date as date 
FROM positions 
WHERE fund_id = :fund_id 
  AND position_date = :position_date  [optional]
```



In [ ]:

positions = db.query_positions(ENGINE, FUND_ID, VALUATION_DATE)
prior_positions = db.query_positions(ENGINE, FUND_ID, PRIOR_BUSINESS_DATE)

position_snapshot = summarise_position_snapshot(positions, prior_positions)

display(position_snapshot["position_summary"])

In [ ]:
display(position_snapshot["asset_class_breakdown"])

In [ ]:
display(position_snapshot["instrument_changes"])

## 2. Price validation

Validate fund administrator prices against Bloomberg prices for liquid instruments.

Price differences can occur for bonds and instruments with limited trading activity. Any difference above the tolerance threshold should be investigated before the dataset is used for risk calculations.

Tolerance thresholds used in this notebook:

- equities and FX: 0.50%
- bonds: 0.25%
- instruments without Bloomberg prices: manual valuation review

<small>

Background call:

```
bbg.bdp(ticker, "PX_LAST", date=valuation_date)
```

In [ ]:
price_validation, flagged_prices = validate_prices(positions, BBG, VALUATION_DATE)
cash_price_check = build_cash_price_check(positions)

display(price_validation)

In [ ]:
flagged_prices = price_validation.loc[price_validation["status"] != "OK"]
display(flagged_prices)


In this simulation, fund administrator prices and Bloomberg prices come from the same mock data source, so differences are not expected. In production, price differences are reviewed by instrument type. Listed equities and exchange-traded derivatives are usually checked against observable market prices. Bonds may require review against evaluated prices, matrix pricing, or dealer marks. Illiquid instruments are validated through fund administrator files, valuation agents, or internal valuation models.

## 3. New instrument onboarding

New instruments are identified by comparing ISINs in `positions` against `prior_positions`. When a new ISIN appears, the workflow pulls Bloomberg reference data where available and checks whether the classification is consistent with the fund administrator file. Bond duration is also checked against remaining maturity. If no new instruments are detected, the notebook displays the standard onboarding controls for reference.

<small>

Background call:

```
bbg.bdp(
    ticker,
    [
        "NAME",
        "ASSET_CLASS",
        "CRNCY",
        "DUR_ADJ_MID",
        "CONVEXITY",
        "YLD_YTM_MID",
        "BETA",
        "RTG_SP",
        "VOLUME_AVG_20D",
    ],
)
```

In [ ]:
onboarding = build_onboarding_review(
    positions,
    position_snapshot["new_isins"],
    BBG,
    VALUATION_DATE,
)

## 4. Market context

Review market conditions before interpreting risk results.

The notebook checks VIX, bond yield moves, and credit spreads for instruments held in the portfolio. These controls help explain whether risk metrics are being calculated in a normal or stressed market environment. See the original Blomberg like calls to bdp an dbdh here.


<small>

Background calls:

```
bbg.bdp("VIX Index", "PX_LAST")

bbg.bdp(bond_tickers, "YLD_YTM_MID")

bbg.bdp(corp_tickers, ["Z_SPRD_MID", "RTG_SP"])
```

In [ ]:
market_review = build_market_context(
    positions,
    BBG,
    VALUATION_DATE,
    PRIOR_BUSINESS_DATE,
)

display(market_review["market_context"])

In [ ]:
display(market_review["yield_moves"])

In [ ]:
display(market_review["credit_spreads"])

## 5. Exception review

Review automated data exceptions before the enriched dataset is used by downstream risk notebooks.

The checks cover:

- price outliers based on recent Bloomberg price history
- missing sensitivities on liquid instruments before enrichment
- single-position concentration above 10% of NAV

<small>

Background calls:


```
SELECT *, position_date as date 
FROM positions 
WHERE fund_id = :fund_id 
  AND position_date = :position_date  [optional]
```


```
bbg.bdh(ticker, "PX_LAST", lookback_start_date, valuation_date)
```



In [ ]:
from src.data.enrichment import query_enriched
positions_enriched = query_enriched(ENGINE, FUND_ID, VALUATION_DATE)

nav = position_snapshot["nav"] 
exception_review = build_exception_review(
    positions_enriched,
    BBG,
    LOOKBACK_START_DATE,
    VALUATION_DATE,
    nav,
)

display(exception_review["price_outliers"])

In [ ]:
# an empty df - no missing sensitivites
display(exception_review["missing_sensitivities"])

In [ ]:
display(exception_review["large_positions"])

In [ ]:
display(exception_review["exception_summary"])

## 6. Risk-ready dataset

Positions are enriched with Bloomberg-style reference data and sensitivities after price validation and exception review.

Liquid instruments are enriched through Bloomberg-style `bdp` calls. Illiquid instruments remain on fund administrator data where Bloomberg fields are not available.

The resulting `risk_df` is the valuation-date dataset used by downstream risk notebooks.


<small>

Background call:

```
FROM positions p
LEFT JOIN positions_enriched e
  ON p.fund_id = e.fund_id AND p.position_date = e.position_date AND p.isin = e.isin

SELECT:
  -- Raw position columns (from positions table)
  p.fund_id
  p.position_date as date
  p.isin
  ...

  -- Enriched columns (from positions_enriched table)
  e.beta                   
  e.dur_adj_mid            
  e.convexity              
  ...

WHERE p.fund_id = ? AND p.position_date = ?
ORDER BY ABS(p.market_value_eur) DESC

```


In [ ]:
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
risk_ready_summary = summarise_risk_ready_dataset(risk_df)
display(risk_ready_summary["enrichment_summary"])

In [ ]:
display(risk_ready_summary["sensitivity_coverage"])

In [ ]:
display(risk_ready_summary["top_positions"])